In [ ]:
# CELL 1: Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

Mounted at /content/drive
✅ Google Drive mounted successfully!


In [ ]:
# CELL 2 : Package File: XAI Libraries, Install timm
import timm
!pip install -q grad-cam pytorch-grad-cam
!pip install -q timm
!pip install -q grad-cam
!pip install -q pytorch-grad-cam
!pip install -q torch-geometric

print("✅ XAI libraries installed!")
print(f"✅ timm installed: {timm.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 80.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement pytorch-grad-cam (from versions: none)
ERROR: No matching distribution found for pytorch-grad-cam
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement pytorch-grad-cam (from versions: none)
ERROR: No matching distribution found for pytorch-grad-cam
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.5 MB/s eta 0:00:00
✅ XAI libraries installed!
✅ timm installed: 1.0.26


In [ ]:
# CELL 3: Import All Libraries and Fix Random Seeds

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from sklearn.preprocessing import label_binarize
from torchvision import transforms
import timm
import time
import shap
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.manifold import TSNE

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
import torchvision.models as models

#GradCAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Image processing
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    auc,
    roc_curve
)
from sklearn.model_selection import train_test_split, StratifiedKFold

# Visualization
from matplotlib.gridspec import GridSpec
# ========== FIX ALL RANDOM SEEDS ==========
def seed_everything(seed=42):
    """Fix all random seeds for reproducibility"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ All random seeds fixed to {seed}")

# Fix seeds
SEED = 42 #42,123,456,
seed_everything(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✅ All libraries imported successfully!")
print("✅ Seeds fixed - Results will be reproducible")
print("✅ SHAP and additional libraries imported!")

✅ All random seeds fixed to 42
✅ Using device: cuda
   GPU: Tesla T4
   CUDA Version: 12.8

✅ All libraries imported successfully!
✅ Seeds fixed - Results will be reproducible
✅ SHAP and additional libraries imported!


In [ ]:
# CELL 4: Fix Base Path
# ============================================

BASE_PATH = '/content/drive/MyDrive/GNN Liver'  # Capital L ✅

CLINICAL_CSV    = f'{BASE_PATH}/Predict Liver Disease/Liver_disease_data.csv'
MRI_DIR         = f'{BASE_PATH}/Liver Cancer Multiclass Dataset'
CT_DIR          = f'{BASE_PATH}/Liver and Liver Tumor Segmentation/dataset'
CT_CSV          = f'{BASE_PATH}/Liver and Liver Tumor Segmentation/lits_df.csv'

RESULTS_PATH    = f'{BASE_PATH}/results'
os.makedirs(RESULTS_PATH, exist_ok=True)

EMBEDDINGS_PATH = f'{RESULTS_PATH}/embeddings'
MODELS_PATH     = f'{RESULTS_PATH}/models'
FIGURES_PATH    = f'{RESULTS_PATH}/figures'
LOGS_PATH       = f'{RESULTS_PATH}/logs'

for path in [EMBEDDINGS_PATH, MODELS_PATH, FIGURES_PATH, LOGS_PATH]:
    os.makedirs(path, exist_ok=True)

MRI_CLASSES = [
    'Angiosarcoma',
    'Cholangiocarcinoma',
    'Healthy',
    'Hemangioma',
    'Hepatocellular Carcinoma'
]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(MRI_CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(MRI_CLASSES)

MRI_IMG_SIZE = 224
CT_IMG_SIZE  = 256
BATCH_SIZE   = 32
NUM_WORKERS  = 2

print(f"✅ Base path fixed: {BASE_PATH}")

# Now check actual subfolder names
print("\n📁 Contents of GNN Liver:")
for item in sorted(os.listdir(BASE_PATH)):
    print(f"   '{item}'")

print("\n📁 Contents of MRI folder:")
for item in sorted(os.listdir(MRI_DIR)):
    print(f"   '{item}'")

print("\n📁 Contents of Predict Liver Disease:")
for item in sorted(os.listdir(f'{BASE_PATH}/Predict Liver Disease')):
    print(f"   '{item}'")

print("\n📁 Contents of CT folder:")
for item in sorted(os.listdir(f'{BASE_PATH}/Liver and Liver Tumor Segmentation')):
    print(f"   '{item}'")

✅ Base path fixed: /content/drive/MyDrive/GNN Liver

📁 Contents of GNN Liver:
   'Liver Cancer Multiclass Dataset'
   'Liver and Liver Tumor Segmentation'
   'Predict Liver Disease'
   'results'

📁 Contents of MRI folder:
   'Angiosarcoma'
   'Cholangiocarcinoma'
   'Healthy'
   'Hemangioma'
   'Hepatocellular Carcinoma'

📁 Contents of Predict Liver Disease:
   'Liver_disease_data.csv'

📁 Contents of CT folder:
   'dataset'
   'lits_df.csv'
   'lits_probe.csv'
   'lits_test.csv'
   'lits_train.csv'


In [ ]:
# CELL 5: Dataset Verification (CT and MRI)
# ============================================

print("=" * 50)
print("📊 DATASET VERIFICATION")
print("=" * 50)

# ── 2. MRI Dataset ───────────────────────────
print("\n📁 Dataset 3 — MRI:")
total_mri = 0
for cls in MRI_CLASSES:
    cls_path = os.path.join(MRI_DIR, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    total_mri += count
    print(f"   ✅ {cls}: {count} images")
print(f"   Total: {total_mri} images")

# ── 3. CT Dataset — CSV only ─────────────────
print("\n📁 Dataset 1 — CT (LiTS17):")
df_ct = pd.read_csv(CT_CSV)
print(f"   ✅ Total slices : {df_ct.shape[0]}")
print(f"   Liver present  : {(~df_ct['liver_mask_empty']).sum()}")
print(f"   Tumor present  : {(~df_ct['tumor_mask_empty']).sum()}")
print(f"   Note: CT folder verified via CSV (Drive I/O skip)")

print("\n" + "=" * 50)
print("✅ Both datasets verified!")
print("=" * 50)

📊 DATASET VERIFICATION

📁 Dataset 6 — Clinical:
   ✅ Rows: 1700, Columns: 11
   Columns: ['Age', 'Gender', 'BMI', 'AlcoholConsumption', 'Smoking', 'GeneticRisk', 'PhysicalActivity', 'Diabetes', 'Hypertension', 'LiverFunctionTest', 'Diagnosis']
   Diagnosis:
Diagnosis
1    936
0    764

📁 Dataset 3 — MRI:
   ✅ Angiosarcoma: 2376 images
   ✅ Cholangiocarcinoma: 2512 images
   ✅ Healthy: 2400 images
   ✅ Hemangioma: 2376 images
   ✅ Hepatocellular Carcinoma: 2328 images
   Total: 11992 images

📁 Dataset 1 — CT (LiTS17):
   ✅ Total slices : 58638
   Liver present  : 39495
   Tumor present  : 51524
   Note: CT folder verified via CSV (Drive I/O skip)

✅ All 3 datasets verified!


In [ ]:
# CELL 6: CNN Backbone — ResNet-50
# ============================================================

print("=" * 50)
print("🏗️  CNN BACKBONE — ResNet-50")
print("=" * 50)

class ResNetEmbeddingModel(nn.Module):
    """
    ResNet-50 backbone for feature extraction.
    Removes classification head → outputs 2048-dim embedding.
    """
    def __init__(self):
        super(ResNetEmbeddingModel, self).__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove final FC layer
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding_dim = 2048

    def forward(self, x):
        x = self.backbone(x)
        x = x.flatten(1)
        return x

# ── Initialize model ──────────────────────────
embedding_model = ResNetEmbeddingModel().to(device)

# ── Count parameters ──────────────────────────
total_params     = sum(p.numel() for p in embedding_model.parameters())
trainable_params = sum(p.numel() for p in embedding_model.parameters()
                       if p.requires_grad)

print(f"\n✅ Model     : ResNet-50")
print(f"✅ Pretrained: ImageNet-1k")
print(f"✅ Total params    : {total_params:,}")
print(f"✅ Trainable params: {trainable_params:,}")
print(f"✅ Device          : {device}")

# ── Quick test ────────────────────────────────
embedding_model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out   = embedding_model(dummy)
    EMBEDDING_DIM = out.shape[1]
    print(f"\n✅ Test forward pass:")
    print(f"   Input  shape : {dummy.shape}")
    print(f"   Output shape : {out.shape}")
    print(f"   Embedding dim: {EMBEDDING_DIM}")

print("\n" + "=" * 50)
print("✅ ResNet-50 ready!")
print("=" * 50)

🏗️  CNN BACKBONE — ResNet-50
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]



✅ Model     : ResNet-50
✅ Pretrained: ImageNet-1k
✅ Total params    : 23,508,032
✅ Trainable params: 23,508,032
✅ Device          : cuda

✅ Test forward pass:
   Input  shape : torch.Size([2, 3, 224, 224])
   Output shape : torch.Size([2, 2048])
   Embedding dim: 2048

✅ ResNet-50 ready!


In [ ]:
# CELL 7: MRI Feature Extraction
# ============================================================

print("=" * 50)
print("🔍 MRI FEATURE EXTRACTION")
print("=" * 50)

embedding_model.eval()

all_embeddings = []
all_labels     = []
all_filepaths  = []

start_time = time.time()

with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(tqdm(embed_loader, desc="Extracting MRI embeddings")):
        imgs = imgs.to(device)
        embs = embedding_model(imgs)
        all_embeddings.append(embs.cpu().numpy())
        all_labels.append(labels.numpy())

        # Progress every 50 batches
        if (batch_idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"   Batch {batch_idx+1}/{len(embed_loader)} | "
                  f"Time: {elapsed:.1f}s")

# ── Stack all ─────────────────────────────────
all_embeddings = np.vstack(all_embeddings)
all_labels     = np.concatenate(all_labels)

elapsed = time.time() - start_time
print(f"\n✅ Extraction complete in {elapsed:.1f}s")
print(f"   Embeddings shape : {all_embeddings.shape}")
print(f"   Labels shape     : {all_labels.shape}")

# ── Save embeddings ───────────────────────────
np.save(f'{EMBEDDINGS_PATH}/mri_embeddings_resnet.npy', all_embeddings)
np.save(f'{EMBEDDINGS_PATH}/mri_labels_resnet.npy',     all_labels)

# Save filepath index
df_embed_index = df_mri.copy()
df_embed_index['embedding_idx'] = range(len(df_mri))
df_embed_index.to_csv(f'{EMBEDDINGS_PATH}/mri_embed_index.csv', index=False)

print(f"\n✅ Saved:")
print(f"   mri_embeddings.npy  → shape {all_embeddings.shape}")
print(f"   mri_labels.npy      → shape {all_labels.shape}")
print(f"   mri_embed_index.csv → {len(df_embed_index)} rows")
print(f"\n📊 Class distribution in embeddings:")
for idx, cls in IDX_TO_CLASS.items():
    count = (all_labels == idx).sum()
    print(f"   {cls}: {count}")

print("\n" + "=" * 50)
print("✅ MRI feature extraction complete!")
print("=" * 50)

🔍 MRI FEATURE EXTRACTION


Extracting MRI embeddings:   0%|          | 0/375 [00:00<?, ?it/s]

   Batch 50/375 | Time: 240.4s
   Batch 100/375 | Time: 425.2s
   Batch 150/375 | Time: 602.7s
   Batch 200/375 | Time: 784.6s
   Batch 250/375 | Time: 948.1s
   Batch 300/375 | Time: 1113.9s
   Batch 350/375 | Time: 1275.0s

✅ Extraction complete in 1355.6s
   Embeddings shape : (11992, 2048)
   Labels shape     : (11992,)

✅ Saved:
   mri_embeddings.npy  → shape (11992, 2048)
   mri_labels.npy      → shape (11992,)
   mri_embed_index.csv → 11992 rows

📊 Class distribution in embeddings:
   Angiosarcoma: 2376
   Cholangiocarcinoma: 2512
   Healthy: 2400
   Hemangioma: 2376
   Hepatocellular Carcinoma: 2328

✅ MRI feature extraction complete!


In [ ]:
# CELL 8: CT Feature Extraction
# ============================================================

print("=" * 50)
print("🔍 CT FEATURE EXTRACTION (LiTS17)")
print("=" * 50)

# ── CT Transform ──────────────────────────────
ct_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── CT Dataset Class ──────────────────────────
class CTDataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.base_dir  = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.loc[idx]
        # Build correct path
        fname    = os.path.basename(row['filepath'])
        img_path = os.path.join(self.base_dir, fname)
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), 0)
        if self.transform:
            img = self.transform(img)
        return img, idx

# ── Load CSV ──────────────────────────────────
df_ct = pd.read_csv(CT_CSV)
print(f"✅ CT CSV loaded: {len(df_ct)} slices")

# Use only slices where liver is present
df_ct_liver = df_ct[~df_ct['liver_mask_empty']].reset_index(drop=True)
print(f"✅ Liver-present slices: {len(df_ct_liver)}")

# Sample max 5000 slices to save time
MAX_CT_SLICES = 5000
if len(df_ct_liver) > MAX_CT_SLICES:
    df_ct_sample = df_ct_liver.sample(
        n=MAX_CT_SLICES, random_state=SEED
    ).reset_index(drop=True)
else:
    df_ct_sample = df_ct_liver.copy()

print(f"✅ Sampled CT slices : {len(df_ct_sample)}")

# ── DataLoader ────────────────────────────────
ct_dataset = CTDataset(df_ct_sample, CT_DIR, transform=ct_transform)
ct_loader  = DataLoader(ct_dataset, batch_size=32,
                        shuffle=False, num_workers=2)

# ── Extract embeddings ────────────────────────
embedding_model.eval()
ct_embeddings = []
start_time    = time.time()

with torch.no_grad():
    for imgs, idxs in tqdm(ct_loader, desc="Extracting CT embeddings"):
        imgs = imgs.to(device)
        embs = embedding_model(imgs)
        ct_embeddings.append(embs.cpu().numpy())

ct_embeddings = np.vstack(ct_embeddings)
elapsed       = time.time() - start_time

print(f"\n✅ CT extraction done in {elapsed:.1f}s")
print(f"   CT embeddings shape: {ct_embeddings.shape}")

# ── Save ──────────────────────────────────────
np.save(f'{EMBEDDINGS_PATH}/ct_embeddings.npy', ct_embeddings)
df_ct_sample.to_csv(f'{EMBEDDINGS_PATH}/ct_embed_index.csv', index=False)

print(f"\n✅ Saved:")
print(f"   ct_embeddings.npy  → shape {ct_embeddings.shape}")
print(f"   ct_embed_index.csv → {len(df_ct_sample)} rows")
print("\n" + "=" * 50)
print("✅ CT feature extraction complete!")
print("=" * 50)

🔍 CT FEATURE EXTRACTION (LiTS17)
✅ CT CSV loaded: 58638 slices
✅ Liver-present slices: 39495
✅ Sampled CT slices : 5000


Extracting CT embeddings:   0%|          | 0/157 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ff369171c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ff369171c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


✅ CT extraction done in 892.4s
   CT embeddings shape: (5000, 2048)

✅ Saved:
   ct_embeddings.npy  → shape (5000, 2048)
   ct_embed_index.csv → 5000 rows

✅ CT feature extraction complete!


In [ ]:
# CELL 9: Patient Similarity Graph Construction
# ============================================================

print("=" * 50)
print("🕸️  PATIENT SIMILARITY GRAPH CONSTRUCTION")
print("=" * 50)

from sklearn.neighbors import kneighbors_graph
from scipy.sparse import csr_matrix
import torch
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix

# ── Load node features ────────────────────────
X = np.load(f'{EMBEDDINGS_PATH}/node_features.npy')
y = np.load(f'{EMBEDDINGS_PATH}/node_labels.npy')

print(f"✅ Node features : {X.shape}")
print(f"✅ Node labels   : {y.shape}")
print(f"✅ Classes       : {np.unique(y)}")

# ── Build KNN graphs ──────────────────────────
K_VALUES = [3, 5, 10]

for k in K_VALUES:
    print(f"\n🔗 Building KNN graph (k={k})...")
    start = time.time()

    # Compute KNN graph
    A = kneighbors_graph(
        X, n_neighbors=k,
        mode='connectivity',
        include_self=True,
        n_jobs=-1
    )

    # Make symmetric
    A = (A + A.T)
    A.data[:] = 1  # Binary edges

    # Convert to PyG format
    edge_index, edge_weight = from_scipy_sparse_matrix(A)

    # Create PyG Data object
    graph_data = Data(
        x          = torch.tensor(X, dtype=torch.float),
        edge_index = edge_index,
        y          = torch.tensor(y, dtype=torch.long)
    )

    elapsed = time.time() - start
    print(f"   ✅ k={k} done in {elapsed:.1f}s")
    print(f"   Nodes : {graph_data.num_nodes}")
    print(f"   Edges : {graph_data.num_edges}")
    print(f"   Features/node: {graph_data.num_node_features}")

    # Save
    save_path = f'{EMBEDDINGS_PATH}/graph_resnet_k{k}.pt'
    torch.save(graph_data, save_path)
    print(f"   Saved: graph_k{k}.pt")

print("\n" + "=" * 50)
print("✅ All KNN graphs constructed and saved!")
print("=" * 50)

🕸️  PATIENT SIMILARITY GRAPH CONSTRUCTION
✅ Node features : (11992, 1034)
✅ Node labels   : (11992,)
✅ Classes       : [0 1 2 3 4]

🔗 Building KNN graph (k=3)...
   ✅ k=3 done in 8.7s
   Nodes : 11992
   Edges : 41486
   Features/node: 1034
   Saved: graph_k3.pt

🔗 Building KNN graph (k=5)...
   ✅ k=5 done in 7.2s
   Nodes : 11992
   Edges : 68898
   Features/node: 1034
   Saved: graph_k5.pt

🔗 Building KNN graph (k=10)...
   ✅ k=10 done in 9.6s
   Nodes : 11992
   Edges : 154538
   Features/node: 1034
   Saved: graph_k10.pt

✅ All KNN graphs constructed and saved!


In [ ]:
# CELL 10: Cosine Graph — Subsampled
# ============================================================

print("=" * 50)
print("🕸️  COSINE SIMILARITY GRAPH (Subsampled)")
print("=" * 50)

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import lil_matrix, eye

X = np.load(f'{EMBEDDINGS_PATH}/node_features.npy')
y = np.load(f'{EMBEDDINGS_PATH}/node_labels.npy')

# Stratified subsample — 400 per class (2000 total)
np.random.seed(SEED)
selected_idx = []
for cls in range(5):
    cls_idx = np.where(y == cls)[0]
    sampled = np.random.choice(cls_idx, size=400, replace=False)
    selected_idx.extend(sampled)

selected_idx = np.array(selected_idx)
X_sub = X[selected_idx]
y_sub = y[selected_idx]

print(f"✅ Subsampled nodes : {X_sub.shape[0]}")
print(f"✅ Per class        : 400 × 5 = 2000")
print(f"✅ Feature dim      : {X_sub.shape[1]}")

THRESHOLDS = [0.7, 0.8]

for thresh in THRESHOLDS:
    print(f"\n🔗 Cosine graph (threshold={thresh})...")
    start = time.time()

    n        = X_sub.shape[0]
    sim_mat  = cosine_similarity(X_sub)
    A        = (sim_mat >= thresh).astype(int)
    np.fill_diagonal(A, 0)

    # To sparse
    A_sparse = csr_matrix(A)
    A_sparse = A_sparse + eye(n, format='csr')
    A_sparse.data[:] = 1

    edge_index, _ = from_scipy_sparse_matrix(A_sparse)

    graph_data = Data(
        x          = torch.tensor(X_sub, dtype=torch.float),
        edge_index = edge_index,
        y          = torch.tensor(y_sub, dtype=torch.long)
    )

    elapsed = time.time() - start
    print(f"   ✅ threshold={thresh} | {elapsed:.1f}s")
    print(f"   Nodes : {graph_data.num_nodes}")
    print(f"   Edges : {graph_data.num_edges}")

    save_path = f'{EMBEDDINGS_PATH}/graph_resnet_cosine_{str(thresh).replace(".","")}.pt'
    torch.save(graph_data, save_path)
    print(f"   Saved: graph_cosine_{str(thresh).replace('.','')}.pt")

# Save subsampled index
np.save(f'{EMBEDDINGS_PATH}/cosine_subset_idx.npy', selected_idx)
print(f"\n✅ Subset index saved")
print("\n" + "=" * 50)
print("✅ Cosine graphs complete!")
print("=" * 50)

🕸️  COSINE SIMILARITY GRAPH (Subsampled)
✅ Subsampled nodes : 2000
✅ Per class        : 400 × 5 = 2000
✅ Feature dim      : 1034

🔗 Cosine graph (threshold=0.7)...
   ✅ threshold=0.7 | 0.5s
   Nodes : 2000
   Edges : 3985718
   Saved: graph_cosine_07.pt

🔗 Cosine graph (threshold=0.8)...
   ✅ threshold=0.8 | 0.5s
   Nodes : 2000
   Edges : 3620640
   Saved: graph_cosine_08.pt

✅ Subset index saved

✅ Cosine graphs complete!


In [ ]:
# CELL 10b: Cosine Graph with Higher Thresholds
# ============================================================

THRESHOLDS = [0.95, 0.99]

for thresh in THRESHOLDS:
    print(f"\n🔗 Cosine graph (threshold={thresh})...")
    start = time.time()

    n       = X_sub.shape[0]
    sim_mat = cosine_similarity(X_sub)
    A       = (sim_mat >= thresh).astype(int)
    np.fill_diagonal(A, 0)

    A_sparse = csr_matrix(A)
    A_sparse = A_sparse + eye(n, format='csr')
    A_sparse.data[:] = 1

    edge_index, _ = from_scipy_sparse_matrix(A_sparse)

    graph_data = Data(
        x          = torch.tensor(X_sub, dtype=torch.float),
        edge_index = edge_index,
        y          = torch.tensor(y_sub, dtype=torch.long)
    )

    elapsed = time.time() - start
    print(f"   ✅ threshold={thresh} | {elapsed:.1f}s")
    print(f"   Nodes : {graph_data.num_nodes}")
    print(f"   Edges : {graph_data.num_edges}")

    save_path = f'{EMBEDDINGS_PATH}/graph_resnet_cosine_{str(thresh).replace(".","")}.pt'
    torch.save(graph_data, save_path)
    print(f"   Saved: graph_cosine_{str(thresh).replace('.','')}.pt")

print("\n✅ Higher threshold cosine graphs done!")


🔗 Cosine graph (threshold=0.95)...
   ✅ threshold=0.95 | 0.1s
   Nodes : 2000
   Edges : 40716
   Saved: graph_cosine_095.pt

🔗 Cosine graph (threshold=0.99)...
   ✅ threshold=0.99 | 0.1s
   Nodes : 2000
   Edges : 3064
   Saved: graph_cosine_099.pt

✅ Higher threshold cosine graphs done!


In [ ]:
# CELL 11: GATv2 Model Definition
# ============================================================

print("=" * 50)
print("🏗️  GAT MODEL DEFINITION")
print("=" * 50)

from torch_geometric.nn import GATv2Conv
import torch.nn.functional as F

class GATv2Classifier(nn.Module):
    def __init__(self, in_channels, hidden_channels,
                 out_channels, heads=4, dropout=0.3):
        super(GATv2Classifier, self).__init__()

        self.dropout = dropout

        # Layer 1
        self.conv1 = GATv2Conv(
            in_channels, hidden_channels,
            heads=heads, dropout=dropout
        )
        # Layer 2
        self.conv2 = GATv2Conv(
            hidden_channels * heads, hidden_channels,
            heads=heads, dropout=dropout
        )
        # Layer 3
        self.conv3 = GATv2Conv(
            hidden_channels * heads, out_channels,
            heads=1, concat=False, dropout=dropout
        )

        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.bn2 = nn.BatchNorm1d(hidden_channels * heads)

    def forward(self, x, edge_index):
        # Layer 1
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Layer 2
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Layer 3 — output
        x = self.conv3(x, edge_index)
        return x

    def get_embeddings(self, x, edge_index):
        """Return layer 2 embeddings for t-SNE"""
        with torch.no_grad():
            x = self.conv1(x, edge_index)
            x = self.bn1(x)
            x = F.elu(x)
            x = self.conv2(x, edge_index)
            x = self.bn2(x)
            x = F.elu(x)
        return x

# ── Initialize model ──────────────────────────
IN_CHANNELS     = 1034   # 1024 MRI + 10 clinical
HIDDEN_CHANNELS = 256
OUT_CHANNELS    = 5      # 5 cancer classes
HEADS           = 4
DROPOUT         = 0.3

model_gat = GATv2Classifier(
    in_channels     = IN_CHANNELS,
    hidden_channels = HIDDEN_CHANNELS,
    out_channels    = OUT_CHANNELS,
    heads           = HEADS,
    dropout         = DROPOUT
).to(device)

total_params = sum(p.numel() for p in model_gat.parameters())
print(f"✅ Model     : GATv2 (3 layers, {HEADS} heads)")
print(f"✅ Input dim : {IN_CHANNELS}")
print(f"✅ Hidden    : {HIDDEN_CHANNELS} × {HEADS} heads")
print(f"✅ Output    : {OUT_CHANNELS} classes")
print(f"✅ Dropout   : {DROPOUT}")
print(f"✅ Params    : {total_params:,}")
print(f"✅ Device    : {device}")

print("\n" + "=" * 50)
print("✅ GATv2 model ready!")
print("=" * 50)

🏗️  GAT MODEL DEFINITION
✅ Model     : GATv2 (3 layers, 4 heads)
✅ Input dim : 1034
✅ Hidden    : 256 × 4 heads
✅ Output    : 5 classes
✅ Dropout   : 0.3
✅ Params    : 4,237,332
✅ Device    : cuda

✅ GATv2 model ready!


In [ ]:
# CELL 12: GATv2 Training — Inductive Setting
# ============================================================

from torch_geometric.utils import subgraph
from sklearn.metrics import roc_auc_score

def train_gat(graph_path, graph_name, epochs=300, lr=0.001):
    print("=" * 50)
    print(f"🚀 TRAINING: {graph_name}")
    print("=" * 50)

    # ── Load graph ────────────────────────────
    graph = torch.load(graph_path, map_location=device, weights_only=False)
    n     = graph.num_nodes

    print(f"✅ Graph loaded: {n} nodes, {graph.num_edges} edges")

    # ── Train/Val/Test split ──────────────────
    idx       = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)

    train_idx = idx[:train_end]
    val_idx   = idx[train_end:val_end]
    test_idx  = idx[val_end:]

    print(f"   Train: {len(train_idx)} | "
          f"Val: {len(val_idx)} | "
          f"Test: {len(test_idx)}")

    # ── Build inductive subgraphs ─────────────
    edge_index_cpu = graph.edge_index.cpu()
    # Train subgraph
    train_edge_index, _ = subgraph(
        train_idx.cpu(), edge_index_cpu,
        relabel_nodes=True, num_nodes=n
    )
    train_graph = Data(
        x          = graph.x[train_idx],
        edge_index = train_edge_index,
        y          = graph.y[train_idx]
    ).to(device)

    # Val subgraph
    val_edge_index, _ = subgraph(
        val_idx.cpu(), edge_index_cpu,
        relabel_nodes=True, num_nodes=n
    )
    val_graph = Data(
        x          = graph.x[val_idx],
        edge_index = val_edge_index,
        y          = graph.y[val_idx]
    ).to(device)

    # Test subgraph
    test_edge_index, _ = subgraph(
        test_idx.cpu(), edge_index_cpu,
        relabel_nodes=True, num_nodes=n
    )
    test_graph = Data(
        x          = graph.x[test_idx],
        edge_index = test_edge_index,
        y          = graph.y[test_idx]
    ).to(device)

    print(f"✅ Inductive subgraphs created!")
    print(f"   Train edges: {train_graph.num_edges}")
    print(f"   Val edges  : {val_graph.num_edges}")
    print(f"   Test edges : {test_graph.num_edges}")

    # ── Model ─────────────────────────────────
    model = GATv2Classifier(
        in_channels     = graph.num_node_features,
        hidden_channels = 256,
        out_channels    = NUM_CLASSES,
        heads           = 4,
        dropout         = 0.4
    ).to(device)

    # Class weights
    labels_np     = graph.y.cpu().numpy()
    class_counts  = np.bincount(labels_np)
    class_weights = torch.tensor(
        1.0 / class_counts, dtype=torch.float
    ).to(device)
    class_weights = class_weights / class_weights.sum()

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-3
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # ── History ───────────────────────────────
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss'  : [], 'val_acc'  : []
    }

    best_val_acc = 0
    best_epoch   = 0
    patience_cnt = 0
    PATIENCE     = 50

    # ── Training loop ─────────────────────────
    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        optimizer.zero_grad()
        out  = model(train_graph.x, train_graph.edge_index)
        loss = criterion(out, train_graph.y)
        loss.backward()
        optimizer.step()
        train_acc = (out.argmax(dim=1) ==
                     train_graph.y).float().mean().item()

        # Validate
        model.eval()
        with torch.no_grad():
            val_out  = model(val_graph.x, val_graph.edge_index)
            val_loss = criterion(val_out, val_graph.y).item()
            val_acc  = (val_out.argmax(dim=1) ==
                        val_graph.y).float().mean().item()

        scheduler.step(val_acc)

        history['train_loss'].append(loss.item())
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch   = epoch
            patience_cnt = 0
            torch.save(
                model.state_dict(),
                f'{MODELS_PATH}/gat_{graph_name}_best.pth'
            )
        else:
            patience_cnt += 1

        if epoch % 20 == 0:
            print(f"   Epoch {epoch:03d} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Train Acc: {train_acc:.4f} | "
                  f"Val Acc: {val_acc:.4f} | "
                  f"Best: {best_val_acc:.4f}")

        if patience_cnt >= PATIENCE:
            print(f"\n⚠️  Early stopping at epoch {epoch}")
            break

    print(f"\n✅ Best Val Acc: {best_val_acc:.4f} at epoch {best_epoch}")

    # ── Test evaluation ───────────────────────
    model.load_state_dict(
        torch.load(
            f'{MODELS_PATH}/gat_{graph_name}_best.pth',
            weights_only=True
        )
    )
    model.eval()
    with torch.no_grad():
        test_out   = model(test_graph.x, test_graph.edge_index)
        test_probs = torch.softmax(test_out, dim=1).cpu().numpy()
        test_preds = test_out.argmax(dim=1).cpu().numpy()
        test_trues = test_graph.y.cpu().numpy()

    test_acc = accuracy_score(test_trues, test_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        test_trues, test_preds, average='macro'
    )
    try:
        test_auc = roc_auc_score(
            label_binarize(test_trues, classes=list(range(NUM_CLASSES))),
            test_probs, multi_class='ovr'
        )
    except:
        test_auc = 0.0

    # GATv2 embeddings for t-SNE
    gat_embeddings = model.get_embeddings(
        test_graph.x, test_graph.edge_index
    ).cpu().numpy()

    print(f"\n📊 TEST RESULTS — {graph_name}:")
    print(f"   Accuracy  : {test_acc:.4f}")
    print(f"   Macro F1  : {f1:.4f}")
    print(f"   AUC-ROC   : {test_auc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"\n{classification_report(test_trues, test_preds, target_names=MRI_CLASSES)}")

    # ── Save everything ───────────────────────
    results = {
        'graph_name'    : graph_name,
        'history'       : history,
        'test_acc'      : test_acc,
        'test_f1'       : f1,
        'test_auc'      : test_auc,
        'test_prec'     : prec,
        'test_rec'      : rec,
        'test_preds'    : test_preds,
        'test_trues'    : test_trues,
        'test_probs'    : test_probs,
        'gat_embeddings': gat_embeddings,
        'best_epoch'    : best_epoch,
        'best_val_acc'  : best_val_acc,
        'train_idx'     : train_idx.numpy(),
        'val_idx'       : val_idx.numpy(),
        'test_idx'      : test_idx.numpy(),
    }

    import joblib
    joblib.dump(results,
                f'{RESULTS_PATH}/gat_{graph_name}_results.pkl')
    np.save(f'{EMBEDDINGS_PATH}/gat_{graph_name}_embeddings.npy',
            gat_embeddings)

    print(f"\n✅ Saved: gat_{graph_name}_results.pkl")
    print(f"✅ Saved: gat_{graph_name}_embeddings.npy")
    print("=" * 50)

    return results

print("✅ Cell 16 (Inductive) ready!")

✅ Cell 16 (Inductive) ready!


In [ ]:
# CELL 13: Train — KNN k=3
# ============================================================

results_k3 = train_gat(
    graph_path = f'{EMBEDDINGS_PATH}/graph_resnet_k3.pt',
    graph_name = 'resnet_knn_k3',
    epochs     = 300,
    lr         = 0.001
)

🚀 TRAINING: resnet_knn_k3
✅ Graph loaded: 11992 nodes, 41486 edges
   Train: 8394 | Val: 1799 | Test: 1799
✅ Inductive subgraphs created!
   Train edges: 22874
   Val edges  : 2513
   Test edges : 2501
   Epoch 020 | Loss: 0.2983 | Train Acc: 0.9093 | Val Acc: 0.9989 | Best: 0.9994
   Epoch 040 | Loss: 0.2285 | Train Acc: 0.9096 | Val Acc: 0.9994 | Best: 1.0000
   Epoch 060 | Loss: 0.2081 | Train Acc: 0.9124 | Val Acc: 0.9989 | Best: 1.0000

⚠️  Early stopping at epoch 77

✅ Best Val Acc: 1.0000 at epoch 27

📊 TEST RESULTS — resnet_knn_k3:
   Accuracy  : 0.9994
   Macro F1  : 0.9994
   AUC-ROC   : 1.0000
   Precision : 0.9994
   Recall    : 0.9995

                          precision    recall  f1-score   support

            Angiosarcoma       1.00      1.00      1.00       337
      Cholangiocarcinoma       1.00      1.00      1.00       364
                 Healthy       1.00      1.00      1.00       354
              Hemangioma       1.00      1.00      1.00       371
Hepatocellul

In [ ]:
# CELL 13b: Train — KNN k=5
# ============================================================

results_k5 = train_gat(
    graph_path = f'{EMBEDDINGS_PATH}/graph_resnet_k5.pt',
    graph_name = 'resnet_knn_k5',
    epochs     = 300,
    lr         = 0.001
)

🚀 TRAINING: resnet_knn_k5
✅ Graph loaded: 11992 nodes, 68898 edges
   Train: 8394 | Val: 1799 | Test: 1799
✅ Inductive subgraphs created!
   Train edges: 36472
   Val edges  : 3143
   Test edges : 3095
   Epoch 020 | Loss: 0.2353 | Train Acc: 0.9348 | Val Acc: 0.9589 | Best: 0.9744
   Epoch 040 | Loss: 0.1715 | Train Acc: 0.9467 | Val Acc: 0.9867 | Best: 0.9872
   Epoch 060 | Loss: 0.1313 | Train Acc: 0.9550 | Val Acc: 0.9867 | Best: 0.9883
   Epoch 080 | Loss: 0.1226 | Train Acc: 0.9615 | Val Acc: 0.9883 | Best: 0.9883
   Epoch 100 | Loss: 0.1181 | Train Acc: 0.9578 | Val Acc: 0.9883 | Best: 0.9889
   Epoch 120 | Loss: 0.1060 | Train Acc: 0.9613 | Val Acc: 0.9872 | Best: 0.9889

⚠️  Early stopping at epoch 131

✅ Best Val Acc: 0.9889 at epoch 81

📊 TEST RESULTS — resnet_knn_k5:
   Accuracy  : 0.9922
   Macro F1  : 0.9922
   AUC-ROC   : 0.9998
   Precision : 0.9922
   Recall    : 0.9923

                          precision    recall  f1-score   support

            Angiosarcoma       0

In [ ]:
# CELL 13c: KNN k=10
results_k10 = train_gat(
    graph_path = f'{EMBEDDINGS_PATH}/graph_resnet_k10.pt',
    graph_name = 'resnet_knn_k10',
    epochs     = 300,
    lr         = 0.001
)

🚀 TRAINING: resnet_knn_k10
✅ Graph loaded: 11992 nodes, 154538 edges
   Train: 8394 | Val: 1799 | Test: 1799
✅ Inductive subgraphs created!
   Train edges: 78068
   Val edges  : 5067
   Test edges : 5053
   Epoch 020 | Loss: 0.2047 | Train Acc: 0.9311 | Val Acc: 0.9455 | Best: 0.9455
   Epoch 040 | Loss: 0.1413 | Train Acc: 0.9519 | Val Acc: 0.9500 | Best: 0.9505
   Epoch 060 | Loss: 0.1088 | Train Acc: 0.9625 | Val Acc: 0.9544 | Best: 0.9550
   Epoch 080 | Loss: 0.0895 | Train Acc: 0.9687 | Val Acc: 0.9466 | Best: 0.9561
   Epoch 100 | Loss: 0.0717 | Train Acc: 0.9741 | Val Acc: 0.9555 | Best: 0.9561
   Epoch 120 | Loss: 0.0655 | Train Acc: 0.9772 | Val Acc: 0.9533 | Best: 0.9561

⚠️  Early stopping at epoch 123

✅ Best Val Acc: 0.9561 at epoch 73

📊 TEST RESULTS — resnet_knn_k10:
   Accuracy  : 0.9572
   Macro F1  : 0.9569
   AUC-ROC   : 0.9970
   Precision : 0.9573
   Recall    : 0.9568

                          precision    recall  f1-score   support

            Angiosarcoma     

In [ ]:
# CELL 13d: Cosine threshold=0.95
results_cos95 = train_gat(
    graph_path = f'{EMBEDDINGS_PATH}/graph_resnet_cosine_095.pt',
    graph_name = 'resnet_cosine_095',
    epochs     = 300,
    lr         = 0.001
)

🚀 TRAINING: resnet_cosine_095
✅ Graph loaded: 2000 nodes, 40716 edges
   Train: 1400 | Val: 300 | Test: 300
✅ Inductive subgraphs created!
   Train edges: 20306
   Val edges  : 1126
   Test edges : 1280
   Epoch 020 | Loss: 0.5803 | Train Acc: 0.7843 | Val Acc: 0.8500 | Best: 0.8500
   Epoch 040 | Loss: 0.4684 | Train Acc: 0.8250 | Val Acc: 0.8967 | Best: 0.8967
   Epoch 060 | Loss: 0.4140 | Train Acc: 0.8486 | Val Acc: 0.8500 | Best: 0.9000
   Epoch 080 | Loss: 0.3900 | Train Acc: 0.8629 | Val Acc: 0.9067 | Best: 0.9067
   Epoch 100 | Loss: 0.3331 | Train Acc: 0.8807 | Val Acc: 0.8933 | Best: 0.9067
   Epoch 120 | Loss: 0.3083 | Train Acc: 0.8857 | Val Acc: 0.8867 | Best: 0.9067

⚠️  Early stopping at epoch 129

✅ Best Val Acc: 0.9067 at epoch 79

📊 TEST RESULTS — resnet_cosine_095:
   Accuracy  : 0.8900
   Macro F1  : 0.8903
   AUC-ROC   : 0.9864
   Precision : 0.8953
   Recall    : 0.8895

                          precision    recall  f1-score   support

            Angiosarcoma   

In [ ]:
# CELL 13e: Cosine threshold=0.99
results_cos99 = train_gat(
    graph_path = f'{EMBEDDINGS_PATH}/graph_resnet_cosine_099.pt',
    graph_name = 'resnet_cosine_099',
    epochs     = 300,
    lr         = 0.001
)

🚀 TRAINING: resnet_cosine_099
✅ Graph loaded: 2000 nodes, 3064 edges
   Train: 1400 | Val: 300 | Test: 300
✅ Inductive subgraphs created!
   Train edges: 1920
   Val edges  : 320
   Test edges : 326
   Epoch 020 | Loss: 0.6797 | Train Acc: 0.6986 | Val Acc: 0.9867 | Best: 0.9867
   Epoch 040 | Loss: 0.6552 | Train Acc: 0.7071 | Val Acc: 0.9967 | Best: 0.9967
   Epoch 060 | Loss: 0.5922 | Train Acc: 0.7129 | Val Acc: 0.9967 | Best: 0.9967
   Epoch 080 | Loss: 0.5684 | Train Acc: 0.7243 | Val Acc: 1.0000 | Best: 1.0000
   Epoch 100 | Loss: 0.5399 | Train Acc: 0.7471 | Val Acc: 0.9933 | Best: 1.0000

⚠️  Early stopping at epoch 112

✅ Best Val Acc: 1.0000 at epoch 62

📊 TEST RESULTS — resnet_cosine_099:
   Accuracy  : 0.9967
   Macro F1  : 0.9967
   AUC-ROC   : 1.0000
   Precision : 0.9966
   Recall    : 0.9968

                          precision    recall  f1-score   support

            Angiosarcoma       0.98      1.00      0.99        57
      Cholangiocarcinoma       1.00      1.00 